In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# function to read directly printed jacobian - inviscid part
def read_direct_jacobian(filename):
    """
    Reads jacobian.txt and reshapes it to (N, 5, 5)
    Assumes first column is index.
    """
    J_raw = pd.read_csv(
        filename,
        delim_whitespace=True,
        header=None
    ).to_numpy()

    # Drop index column and reshape
    J = J_raw[:, 1:].reshape(-1, 5, 5, order="F")
    return J

J_direct_inviscid = read_direct_jacobian("jacobian.txt")

In [1]:
# compute inviscid jacobian by own method

def get_exact_inviscid_jacobian_rhoE(U, nx, ny, nz, gamma):
    """
    Computes the exact analytical 3D Euler flux Jacobian matrix 
    where the conservative state vector is U = [rho, rhou, rhov, rhow, rhoE].
    """
    rho  = U[0]
    rhou = U[1]
    rhov = U[2]
    rhow = U[3]
    rhoE = U[4]
    
    u = rhou / rho
    v = rhov / rho
    w = rhow / rho
    
    q2 = u**2 + v**2 + w**2
    # p = (rhoE - 0.5 * rho * q2) * (gamma - 1)
    p = (rhoE - 0.5 * (rhou*u + rhov*v + rhow*w)) * (gamma - 1.0)
    
    V_n = u * nx + v * ny + w * nz
    g1 = gamma - 1.0
    
    A = np.zeros((5, 5))
    
    # Row 0: Continuity (rho)
    A[0, 0] = 0.0
    A[0, 1] = nx
    A[0, 2] = ny
    A[0, 3] = nz
    A[0, 4] = 0.0
    
    # Row 1: X-Momentum (rhou)
    A[1, 0] = -u * V_n + 0.5 * g1 * q2 * nx
    A[1, 1] = V_n + (2.0 - gamma) * u * nx
    A[1, 2] = u * ny - g1 * v * nx
    A[1, 3] = u * nz - g1 * w * nx
    A[1, 4] = g1 * nx
    
    # Row 2: Y-Momentum (rhov)
    A[2, 0] = -v * V_n + 0.5 * g1 * q2 * ny
    A[2, 1] = v * nx - g1 * u * ny
    A[2, 2] = V_n + (2.0 - gamma) * v * ny
    A[2, 3] = v * nz - g1 * w * ny
    A[2, 4] = g1 * ny
    
    # Row 3: Z-Momentum (rhow)
    A[3, 0] = -w * V_n + 0.5 * g1 * q2 * nz
    A[3, 1] = w * nx - g1 * u * nz
    A[3, 2] = w * ny - g1 * v * nz
    A[3, 3] = V_n + (2.0 - gamma) * w * nz
    A[3, 4] = g1 * nz
    
    # Row 4: Energy (rhoE) -> Notice the changes here due to d(rhoE) mapping
    A[4, 0] = V_n * (g1 * q2 - (gamma * rhoE / rho))
    A[4, 1] = (gamma * rhoE / rho - 0.5 * g1 * q2) * nx - g1 * u * V_n
    A[4, 2] = (gamma * rhoE / rho - 0.5 * g1 * q2) * ny - g1 * v * V_n
    A[4, 3] = (gamma * rhoE / rho - 0.5 * g1 * q2) * nz - g1 * w * V_n
    A[4, 4] = gamma * V_n
    
    return A